# 04  --  Forecasting the Energy Transition

The first three notebooks *described* what happened: renewable share tripled, fossil consumption fell, dependency barely moved. This notebook asks a forward-looking question instead --  **can we predict where these metrics are heading?**  --  and uses it to build a first, honest machine-learning workflow on data we already understand deeply.

**Why extend this project with ML rather than start fresh:** the modelling is only half the job. The harder half is knowing whether a forecast is trustworthy, and that judgement comes from understanding the data. We already know this data's quirks (the 2020 COVID dip, the 2022 gas crisis, the slow grind of dependency), so we can tell a good forecast from a lucky one.

---

### What this notebook does

We forecast **two** EU-level metrics and deliberately compare the results:

| Target | What we expect |
|---|---|
| **Renewable share (%)** | Strong, steady upward trend  --  should be predictable |
| **Energy dependency rate (%)** | Flat and noisy  --  should resist prediction |

The contrast is the lesson. A model can only *surface* signal that exists in the data; it cannot *manufacture* it. Renewable share has a trend a simple model can ride; dependency does not  --  and honest evaluation is what tells the two apart. That's the same conclusion the analysis reached, now expressed in the language of prediction error.

### The ML concepts introduced here

1. **Framing** a time series as a supervised learning problem
2. **Train / test splitting in time order**  --  and why shuffling is a trap for time series
3. **Baselines first**  --  a model only earns its keep by beating a naive guess
4. **Evaluation metrics**  --  MAE, RMSE, MAPE, and what each one tells you
5. **Overfitting**  --  why a model that fits the past better can forecast the future worse
6. **Forecasting with honest uncertainty**

> **A caveat stated up front:** we have **20 annual data points**. That is tiny. No amount of modelling sophistication overcomes 20 observations, so we stay deliberately simple  --  and treat the *workflow* and the *judgement*, not the model, as the thing worth learning.

In [15]:
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.config import DB_PATH
from utils.charts import COLORS

conn = sqlite3.connect(DB_PATH)

## 1. Load the data we already cleaned

We reuse the exact same SQL that powers notebook 03 and the dashboards  --  no new data, no new cleaning. The EU-wide **renewable share** is renewable primary production as a percentage of gross available energy; the **dependency rate** is the average across the 27 member states of (imports − exports) / gross available energy.

Pulling these from the same source the rest of the project uses means the forecast sits on numbers we've already validated.

In [16]:
eu = pd.read_sql('''
    WITH eu_renew AS (
        SELECT year, SUM(value_gwh) AS renewable_gwh FROM renewables
        WHERE energy_source = 'Renewables and biofuels' AND balance_type = 'Primary production'
        GROUP BY year
    ),
    eu_gae AS (
        SELECT year, SUM(value_gwh) AS gae FROM energy_dependency
        WHERE balance_type = 'Gross available energy' AND energy_source = 'Total'
        GROUP BY year
    ),
    shares AS (
        SELECT r.year, ROUND(r.renewable_gwh / NULLIF(g.gae, 0) * 100, 2) AS renewable_share
        FROM eu_renew r JOIN eu_gae g ON r.year = g.year
    ),
    country_dep AS (
        SELECT year, country,
            (SUM(CASE WHEN balance_type = 'Imports' THEN value_gwh ELSE 0 END)
             - SUM(CASE WHEN balance_type = 'Exports' THEN value_gwh ELSE 0 END))
            / NULLIF(SUM(CASE WHEN balance_type = 'Gross available energy' THEN value_gwh ELSE 0 END), 0) * 100 AS dependency_rate
        FROM energy_dependency WHERE energy_source = 'Total'
        GROUP BY year, country
    ),
    eu_dep AS (
        SELECT year, ROUND(AVG(dependency_rate), 2) AS dependency_rate
        FROM country_dep GROUP BY year
    )
    SELECT s.year, s.renewable_share, d.dependency_rate
    FROM shares s JOIN eu_dep d ON s.year = d.year
    ORDER BY s.year
''', conn)

print(f"{len(eu)} years: {eu['year'].min()}–{eu['year'].max()}")
eu

20 years: 2005–2024


,year,renewable_share,dependency_rate
0,2005,7.16,57.45
1,2006,7.51,58.52
2,2007,8.18,57.95
3,2008,8.84,58.87
4,2009,9.81,56.72
5,2010,10.51,55.89
6,2011,10.61,56.90
7,2012,11.94,56.02
8,2013,12.81,55.50
9,2014,13.26,55.01


Before modelling anything, look at the two series side by side. This single chart is the whole notebook in miniature: one line marches steadily upward, the other wanders sideways. Any honest forecaster forms an expectation *here*, before fitting a thing.

In [17]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=eu['year'], y=eu['renewable_share'],
                         name='Renewable share', mode='lines+markers',
                         line=dict(color=COLORS['renewables'], width=2.5)))
fig.add_trace(go.Scatter(x=eu['year'], y=eu['dependency_rate'],
                         name='Dependency rate', mode='lines+markers',
                         line=dict(color=COLORS['dependency'], width=2.5)))
fig.update_layout(
    title='The two targets: one trends, one drifts',
    yaxis_title='Percent (%)', xaxis_title=None,
    plot_bgcolor='white', yaxis=dict(gridcolor='#eeeeee'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

## 2. Framing a forecast as supervised learning

Machine-learning models learn a mapping from **features** (`X`) to a **target** (`y`). A time series doesn't arrive in that shape, so we have to *frame* it as one.

The simplest possible framing: use **time itself as the feature**. We create `t`  --  years elapsed since 2005  --  and ask the model to learn `share ≈ f(t)`. Starting at 0 instead of using the raw year (2005, 2006, …) keeps the numbers small and the intercept interpretable as "the 2005 value."

This framing assumes the trend is a function of time alone. That is a strong, simplistic assumption  --  it ignores prices, policy, weather  --  but it is the right *first* model: you cannot tell whether a complicated model is worth it until you know what the simplest one achieves.

In [18]:
eu['t'] = eu['year'] - eu['year'].min()
eu[['year', 't', 'renewable_share', 'dependency_rate']].head()

,year,t,renewable_share,dependency_rate
0,2005,0,7.16,57.45
1,2006,1,7.51,58.52
2,2007,2,8.18,57.95
3,2008,3,8.84,58.87
4,2009,4,9.81,56.72


## 3. Splitting in time order  --  and why shuffling is a trap

To know whether a model can *predict*, we must test it on data it never saw during training. The standard tool is a train/test split.

For most ML problems you split **randomly**. For a time series you must **not**. Forecasting means predicting the *future* from the *past*, so the test set has to be the most recent years. If you shuffled and let the model train on 2024 to help predict 2020, you'd be leaking future information backwards  --  the test score would look great and mean nothing. This mistake (look-ahead leakage) is one of the most common ways a time-series model fools its author.

So we train on **2005–2018** (the first 14 years) and hold out **2019–2024** (the last 6) as a genuine out-of-sample test.

In [19]:
TRAIN_END = 2018
train = eu[eu['year'] <= TRAIN_END]
test  = eu[eu['year'] >  TRAIN_END]
print(f"train: {train['year'].min()}–{train['year'].max()}  ({len(train)} rows)")
print(f"test:  {test['year'].min()}–{test['year'].max()}  ({len(test)} rows)")

fig = go.Figure()
fig.add_vrect(x0=TRAIN_END + 0.5, x1=eu['year'].max() + 0.3,
              fillcolor='#5B8DB8', opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=train['year'], y=train['renewable_share'],
                         name='Train (2005–2018)', mode='lines+markers',
                         line=dict(color=COLORS['renewables'], width=2.5)))
fig.add_trace(go.Scatter(x=test['year'], y=test['renewable_share'],
                         name='Test (2019–2024)', mode='lines+markers',
                         line=dict(color=COLORS['neutral'], width=2.5, dash='dot')))
fig.update_layout(
    title='Time-ordered split: the model never sees the shaded years while training',
    yaxis_title='Renewable share (%)', xaxis_title=None,
    plot_bgcolor='white', yaxis=dict(gridcolor='#eeeeee'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

train: 2005–2018  (14 rows)
test:  2019–2024  (6 rows)


## 4. Always start with a baseline

Before any model, ask: *what's the dumbest reasonable guess?* If a model can't beat that, it is adding complexity for nothing.

For a forecast, the classic baseline is **naive (persistence)**: predict that every future year equals the last value we saw in training (2018). It assumes "tomorrow looks like today"  --  no trend, no cleverness. A model earns its place only by beating this.

We score everything with three error metrics, each answering a different question:

- **MAE** (mean absolute error)  --  *on average, how many percentage points off are we?* Easy to read, in the target's own units.
- **RMSE** (root mean squared error)  --  same units, but squares the errors first, so it **punishes large misses harder**. RMSE > MAE signals a few big errors.
- **MAPE** (mean absolute percentage error)  --  error as a *percentage of the actual value*, for comparing across targets on different scales.

In [20]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def score(y_true, y_pred):
    return {
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAPE': mape(np.asarray(y_true), np.asarray(y_pred)),
    }

# Naive baseline for renewable share: every test year = last training value
last_value = train['renewable_share'].iloc[-1]
naive_pred = np.full(len(test), last_value)

print(f"Naive forecast = {last_value:.2f}% for every year 2019–2024")
pd.DataFrame({'naive': score(test['renewable_share'], naive_pred)}).T.round(2)

Naive forecast = 14.48% for every year 2019–2024


,MAE,RMSE,MAPE
naive,3.03,3.4,16.68


## 5. The first real model: linear regression

A naive forecast ignores the obvious upward march. **Linear regression** fits the straight line `share = intercept + slope · t` that minimises squared error on the training years, then extends it into the test years. One parameter  --  the slope  --  captures "how many percentage points per year."

We fit on **train only**, predict the **held-out** test years, and compare to the naive baseline on the same years.

In [21]:
feat = ['t']
lin = LinearRegression().fit(train[feat], train['renewable_share'])
lin_pred = lin.predict(test[feat])

print(f"Learned trend: {lin.intercept_:.2f}% in 2005, +{lin.coef_[0]:.3f} pp per year")

results = pd.DataFrame({
    'naive':  score(test['renewable_share'], naive_pred),
    'linear': score(test['renewable_share'], lin_pred),
}).T.round(2)
results

Learned trend: 7.27% in 2005, +0.596 pp per year


,MAE,RMSE,MAPE
naive,3.03,3.40,16.68
linear,0.59,0.73,3.23


In [22]:
# Visualise the fit: training line, the held-out truth, and both forecasts
full_line = lin.predict(eu[feat])

fig = go.Figure()
fig.add_vrect(x0=TRAIN_END + 0.5, x1=eu['year'].max() + 0.3,
              fillcolor='#5B8DB8', opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=eu['year'], y=eu['renewable_share'],
                         name='Actual', mode='markers',
                         marker=dict(color=COLORS['renewables'], size=8)))
fig.add_trace(go.Scatter(x=eu['year'], y=full_line,
                         name='Linear fit', line=dict(color=COLORS['renewables'], width=2)))
fig.add_trace(go.Scatter(x=test['year'], y=naive_pred,
                         name='Naive forecast', line=dict(color=COLORS['nuclear'], width=2, dash='dash')))
fig.update_layout(
    title='Linear regression vs naive baseline (shaded = held-out test years)',
    yaxis_title='Renewable share (%)', xaxis_title=None,
    plot_bgcolor='white', yaxis=dict(gridcolor='#eeeeee'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

The linear model cuts test error to roughly a fifth of the naive baseline (MAE ~0.6 vs ~3.0 percentage points). On a metric with a genuine trend, a one-parameter model is hard to beat  --  it rides the slope straight into the test years.

But "fits better" is a seductive idea, and the next section shows how it misleads.

## 6. Overfitting: when fitting the past *better* forecasts the future *worse*

If a straight line is good, surely a more flexible curve is better? A **degree-4 polynomial** can bend to chase every wiggle in the training data. Let's fit one and compare its **training** fit and its **test** forecast against the simple line.

In [23]:
poly = make_pipeline(PolynomialFeatures(degree=4), LinearRegression())
poly.fit(train[feat], train['renewable_share'])

compare = pd.DataFrame({
    'linear  (train R²)': [lin.score(train[feat], train['renewable_share'])],
    'poly-4  (train R²)': [poly.score(train[feat], train['renewable_share'])],
    'linear  (test MAE)':  [score(test['renewable_share'], lin_pred)['MAE']],
    'poly-4  (test MAE)':  [score(test['renewable_share'], poly.predict(test[feat]))['MAE']],
}).T.round(3)
compare.columns = ['value']
compare

,value
linear (train R²),0.970
poly-4 (train R²),0.992
linear (test MAE),0.585
poly-4 (test MAE),0.928


In [24]:
# The polynomial hugs the training points, then lurches once it leaves them
grid = pd.DataFrame({'t': np.linspace(0, eu['t'].max(), 200)})
grid['year'] = grid['t'] + eu['year'].min()

fig = go.Figure()
fig.add_vrect(x0=TRAIN_END + 0.5, x1=eu['year'].max() + 0.3,
              fillcolor='#5B8DB8', opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=eu['year'], y=eu['renewable_share'],
                         name='Actual', mode='markers',
                         marker=dict(color=COLORS['renewables'], size=8)))
fig.add_trace(go.Scatter(x=grid['year'], y=lin.predict(grid[['t']]),
                         name='Linear (degree 1)', line=dict(color=COLORS['neutral'], width=2)))
fig.add_trace(go.Scatter(x=grid['year'], y=poly.predict(grid[['t']]),
                         name='Polynomial (degree 4)', line=dict(color=COLORS['fossil'], width=2)))
fig.update_layout(
    title='Overfitting: the flexible curve fits training points, then mis-forecasts',
    yaxis_title='Renewable share (%)', xaxis_title=None,
    plot_bgcolor='white', yaxis=dict(gridcolor='#eeeeee'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

The polynomial scores a **higher training R²**  --  it fits the years it has seen more snugly  --  yet its **test error is worse**. That gap is the textbook signature of **overfitting**: the extra flexibility memorised the noise in the training years instead of learning the durable trend, and noise doesn't repeat. With only 14 training points, flexibility is a liability, not a feature.

> The discipline this teaches: **judge a model on held-out error, never on how well it fits data it was trained on.**

## 7. The honest contrast: forecasting dependency rate

Everything so far worked because renewable share trends. Now turn the identical pipeline on the **dependency rate** and watch what happens when the signal isn't there. We wrap the workflow in one function so the only thing that changes is the target column.

In [25]:
def evaluate_target(target, label):
    tr, te = eu[eu['year'] <= TRAIN_END], eu[eu['year'] > TRAIN_END]
    naive = np.full(len(te), tr[target].iloc[-1])
    model = LinearRegression().fit(tr[feat], tr[target])
    pred = model.predict(te[feat])
    table = pd.DataFrame({
        'naive':  score(te[target], naive),
        'linear': score(te[target], pred),
    }).T.round(2)
    print(f"{label}: learned slope = {model.coef_[0]:+.3f} pp/year, "
          f"train R² = {model.score(tr[feat], tr[target]):.2f}")
    return table

print('— Renewable share —')
display(evaluate_target('renewable_share', 'Renewable share'))
print('\n— Dependency rate —')
display(evaluate_target('dependency_rate', 'Dependency rate'))

— Renewable share —
Renewable share: learned slope = +0.596 pp/year, train R² = 0.97


,MAE,RMSE,MAPE
naive,3.03,3.40,16.68
linear,0.59,0.73,3.23



— Dependency rate —
Dependency rate: learned slope = -0.142 pp/year, train R² = 0.28


,MAE,RMSE,MAPE
naive,1.75,2.14,2.98
linear,2.61,3.20,4.40


The result flips. For dependency rate the **naive baseline wins**  --  the trend model's test error is *larger* than just guessing last year's value, and the training R² is weak (~0.28 vs ~0.97 for renewable share). There is barely any linear signal to learn, so fitting a slope mostly fits noise, and that hurts out-of-sample.

This is the payoff of the whole notebook, and it is not a failure  --  it is the model telling the truth:

> **Renewable share is predictable because the transition is a real, sustained trend. Dependency rate is not, because  --  exactly as the analysis concluded  --  it is pushed around by domestic production, nuclear policy, demand, and trade all at once, with no single direction.**

The ML restates the project's central finding in a sharper form: *one of these metrics carries signal a simple model can ride into the future, and one does not.* A less careful analyst, skipping the baseline, would have reported the dependency "forecast" as if it meant something.

## 8. Forecasting forward, with honest uncertainty

For the metric that *is* predictable, we now produce an actual forecast. Two correct habits here:

1. **Refit on all 20 years.** The train/test split was for *validating* the method. Once we trust it, we use every data point we have to fit the model we'll actually project with.
2. **Show uncertainty, not just a line.** We draw a band of ±2× the residual standard deviation around the forecast. This is a deliberately rough band  --  with 20 points it is indicative, not a rigorous confidence interval  --  but a forecast shown without *any* uncertainty invites false confidence.

In [26]:
final = LinearRegression().fit(eu[feat], eu['renewable_share'])
resid_std = np.std(eu['renewable_share'] - final.predict(eu[feat]), ddof=2)

future = pd.DataFrame({'year': range(2025, 2031)})
future['t'] = future['year'] - eu['year'].min()
future['forecast'] = final.predict(future[['t']])
future['lower'] = future['forecast'] - 2 * resid_std
future['upper'] = future['forecast'] + 2 * resid_std

print(f"Trend: +{final.coef_[0]:.3f} pp/year   (residual std ≈ {resid_std:.2f} pp)")
future.round(2)

Trend: +0.629 pp/year   (residual std ≈ 0.51 pp)


,year,t,forecast,lower,upper
0,2025,20,19.66,18.64,20.68
1,2026,21,20.29,19.27,21.31
2,2027,22,20.92,19.90,21.94
3,2028,23,21.55,20.52,22.57
4,2029,24,22.17,21.15,23.20
5,2030,25,22.80,21.78,23.82


In [27]:
fig = go.Figure()
# uncertainty band
fig.add_trace(go.Scatter(
    x=list(future['year']) + list(future['year'][::-1]),
    y=list(future['upper']) + list(future['lower'][::-1]),
    fill='toself', fillcolor='rgba(76,175,80,0.15)',
    line=dict(width=0), name='±2 residual std', hoverinfo='skip'))
fig.add_trace(go.Scatter(x=eu['year'], y=eu['renewable_share'],
                         name='Actual (2005–2024)', mode='lines+markers',
                         line=dict(color=COLORS['renewables'], width=2.5)))
fig.add_trace(go.Scatter(x=future['year'], y=future['forecast'],
                         name='Forecast (2025–2030)', mode='lines+markers',
                         line=dict(color=COLORS['renewables'], width=2.5, dash='dash')))
fig.update_layout(
    title='EU renewable share: forecast to 2030',
    yaxis_title='Renewable share (%)', xaxis_title=None,
    plot_bgcolor='white', yaxis=dict(gridcolor='#eeeeee'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

The straight-line trend projects EU renewable share reaching roughly **23% by 2030**. Notice the forecast for 2025 sits slightly *below* the 2024 actual: a single straight line can't capture the recent acceleration (2022–2024 rose faster than the early years), so it splits the difference. That visible bias is itself a finding  --  it hints that a model allowing the slope to steepen might fit better, which is exactly the kind of next step a real forecasting project would test.

## 9. What I'd do next  --  the learning roadmap

This notebook is a *foundation*, not a finished forecasting system. Held honestly, its limits point straight at what to learn next:

**Limitations of what we built**
- **20 annual points** is far too few for any model to generalise strongly. The forecast is a reasoned extrapolation, not a confident prediction.
- **Time as the only feature** ignores every real driver  --  energy prices, policy targets, weather, GDP.
- A **straight line cannot bend**, so it misses the recent acceleration and would eventually project an impossible share above 100%.
- The uncertainty band is a rough rule of thumb, not a calibrated interval.

**Where to go from here**
- **More data, same idea:** use the **country panel** (27 countries × 20 years ≈ 540 rows) to train a regression with real features  --  turning a tiny time series into a respectable supervised-learning dataset. This is the single highest-value next step.
- **Proper time-series models:** `statsmodels` for ARIMA / exponential smoothing, which model autocorrelation directly instead of treating time as a plain feature.
- **Better validation:** **time-series cross-validation** (expanding-window) instead of one split, for a more stable estimate of error.
- **Regularisation & model selection:** Ridge/Lasso and cross-validated polynomial degree, to choose flexibility by evidence rather than by eye.

**The transferable lesson  --  the actual point of this notebook:** the workflow matters more than the model. *Frame the problem, split honestly in time, beat a baseline, evaluate on held-out data, and let the errors tell you which questions the data can and cannot answer.* That discipline is what separates a data scientist from someone who can call `.fit()`.

In [28]:
conn.close()